In [11]:
## Python v3.10.13
import sys
sys.path.append('/Users/williamharrigan/Desktop/test_wagner')
import creds
import numpy as np

## Validation functions
from validation_functions import * 

# ── AI / LLM ─────────────────────────────────────────────────────
from openai import AsyncOpenAI                          # async OpenAI API client
from pydantic import BaseModel, Field, create_model     # data validation and schema definition
from pydantic_ai import Agent                           # high-level LLM agent abstraction
from pydantic_ai.models.openai import OpenAIChatModel   # pydantic-ai wrapper for OpenAI chat models
from pydantic_ai.providers.openai import OpenAIProvider # provider config (base URL, auth) for OpenAI
from enum import Enum

In [9]:
class AreAnntoationsEqual(BaseModel):
    are_equal: bool = Field(..., description="Are the two values synonymous?")
    # justification: str = Field(..., description="Justification of the propose value for are_equal. Justifications should be as concise as possible.")
    

## Setting the prompt and model for validation agent
validation_agent = Agent(
    # model="openai:o3-mini",
    model="openai:o4-mini",
    output_type=AreAnntoationsEqual,
    system_prompt = """You are an expert taxonomist. You are comparing the outcome of a manually extracted result versus an automatically extracted result. You need to compare the automatic results and determine whether the result is synonymous or equal the manual one; taking into consideration
    linguisitc and formatting nuances. If the measurements are correct but are seemigly in the wrong units, you can mark that as the results being the same = True. Your answer is whether the two results are similar True/False.""",
)

In [12]:
rows_to_validate = pd.read_csv('../extraction_agent/extracted_wagner_data.csv')
rows_to_validate = rows_to_validate.replace('[]', np.nan)
rows_to_validate = rows_to_validate.replace('set()', np.nan)
rows_to_validate.head()

/var/folders/6n/xxr1dffx2lbdwlqml4kf1yy40000gn/T/ipykernel_75133/2931612778.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rows_to_validate = rows_to_validate.replace('[]', np.nan)


,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,breeding_type,...,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Apiaceae,Daucus,pusillus,American carrot,0,Dicots,NaN,['HISPID'],NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
1,Apiaceae,Daucus,pusillus,American carrot,pg 203-204,Dicots,NaN,PUBERULENT,ALTERNATE,MONOECIOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
2,Apiaceae,Hydrocotyle,bowlesioides,Marsh pennywort,0,Dicots,NaN,['HIRSUTE'],NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides
3,Apiaceae,Hydrocotyle,bowlesioides,marsh pennywort,pg 205-206,Dicots,NaN,HIRSUTE,ALTERNATE,MONOECIOUS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides
4,Oleaceae,Nestegis,sandwicensis,Olive family,76,Dicots,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,oleaceae_nestegis_sandwicensis


In [4]:
def _is_empty(val):
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    if isinstance(val, str) and val.strip().lower() in ("nan", "none", ""):
        return True
    return False

async def _compare_column(col, auto_val, manual_val):
    """Return (col, code): 0=incorrect, 1=correct, 2=needs manual review."""
    auto_empty = _is_empty(auto_val)
    manual_empty = _is_empty(manual_val)

    if auto_empty and manual_empty:
        return col, 1
    if auto_empty != manual_empty:
        return col, 2

    result = await validation_agent.run(f"Manual: {manual_val} Automatic: {auto_val}")
    return col, (1 if result.output.are_equal else 0)

async def build_validation_df(df, skip_cols=None):
    """
    Compare auto vs manual rows and return a 3-row-per-species_key DataFrame.

    Input df: 2 rows per species_key — row 1 = auto extracted, row 2 = manual extracted.

    Output row order per species_key:
      1. manual extracted   (row_type = "Manual")
      2. auto extracted     (row_type = "Automatic")
      3. coded comparison   (row_type = "IsCorrect"; values: 0=incorrect, 1=correct, 2=needs review)

    Args:
        df:         DataFrame with 2 rows per species_key
        skip_cols:  Columns excluded from agent comparison (copied as-is to coded row)
    """
    if skip_cols is None:
        skip_cols = {"species_key"}

    compare_cols = [c for c in df.columns if c not in skip_cols]
    result_rows = []

    for species_key, group in df.groupby("species_key", sort=False):
        if len(group) != 2:
            print(f"Warning: {species_key} has {len(group)} rows, expected 2 — skipping.")
            continue

        auto_row   = group.iloc[0]
        manual_row = group.iloc[1]

        col_results = await asyncio.gather(
            *[_compare_column(col, auto_row[col], manual_row[col]) for col in compare_cols]
        )

        coded_row = {col: None for col in df.columns}
        for col in skip_cols:
            if col in df.columns:
                coded_row[col] = auto_row[col]
        for col, code in col_results:
            coded_row[col] = code

        result_rows.extend([
            {"row_type": "Manual",     **manual_row.to_dict()},
            {"row_type": "Automatic",  **auto_row.to_dict()},
            {"row_type": "IsCorrect",  **coded_row},
        ])

    out_cols = ["row_type"] + df.columns.tolist()
    return pd.DataFrame(result_rows, columns=out_cols)

In [5]:
# Run comparison
validation_df = await build_validation_df(rows_to_validate)
validation_df.head()

,row_type,family,genus,species,common_name,wagner_pg_number,description,infraspecific_epithet,stem_hair_type,phyllotaxy_type,...,seeds_perfruit,seed_length,seed_width,seed_diameter,pistillate_peduncle_length,pistillate_peduncle_width,staminate_pedicel_length,staminate_peduncle_length,staminate_peduncle_width,species_key
0,Manual,Apiaceae,Daucus,pusillus,American carrot,pg 203-204,Dicots,NaN,PUBERULENT,ALTERNATE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
1,Automatic,Apiaceae,Daucus,pusillus,American carrot,0,Dicots,NaN,['HISPID'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_daucus_pusillus
2,IsCorrect,1,1,1,1,0,1,1,0,0,...,1.0,1,1,1,1.0,1.0,1,1.0,1.0,apiaceae_daucus_pusillus
3,Manual,Apiaceae,Hydrocotyle,bowlesioides,marsh pennywort,pg 205-206,Dicots,NaN,HIRSUTE,ALTERNATE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides
4,Automatic,Apiaceae,Hydrocotyle,bowlesioides,Marsh pennywort,0,Dicots,NaN,['HIRSUTE'],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,apiaceae_hydrocotyle_bowlesioides


In [6]:
### Output validation results to an excel sheet 

export_validation_to_excel(validation_df, 'validation_output.xlsx')

Saved: validation_output.xlsx
